In [14]:
import sys
import os
from pathlib import Path
import pathlib
import pandas as pd
import numpy as np
import yaml

In [15]:
# Detecção automática de caminhos
notebook_path = pathlib.Path().absolute()
root_folder = notebook_path.parent

# Definir caminhos
SRC_DIR = root_folder / "src/data_processing"
CONFIG_DIR = root_folder / "config"
DATA_DIR = root_folder / "data"
COMPLETE_SERIES_DIR = DATA_DIR / "complete_series"
PROCESSED_DIR = DATA_DIR / "processed"
FORECAST_DIR = DATA_DIR / "forecast"
COMPLETE_SERIES_INFERENCE_DIR = DATA_DIR / "complete_series_inference"
PROCESSED_INFERENCE_DIR = DATA_DIR / "processed_inference"
FORECAST_INFERENCE_DIR = DATA_DIR / "forecast_inference"                   

# Adicionar src ao path
sys.path.append(str(SRC_DIR))

In [16]:
# Configuração de caminhos
root_folder = Path.cwd().parent # Ajuste conforme sua estrutura
sys.path.insert(0, str(root_folder / "src"))

from utils.config_loader import ConfigLoader
from data_processing.features_processing import process_features

# Inicializar Loader
loader = ConfigLoader(config_path=root_folder / "config" / "data_config.yaml")
data_config = loader.load_full_config()

# Extrair variáveis globais do config
stations = data_config["stations"]
flow_window_config = data_config["flow_window_config"]
climate_window_config = data_config["climate_window_config"]
temporal_features = data_config["temporal_features"]
static_attributes_dict = data_config["static_attributes"]
horizon = data_config["horizon"]

print(f"✅ Configurações carregadas para as estações: {stations}")

ImportError: cannot import name 'process_inference_simple' from 'data_processing.features_processing_inference' (c:\Users\emily\Documents\GitHub\Mini-curso-GitHub-Leo\src\data_processing\features_processing_inference.py)

In [17]:
# Mapeamento de colunas para os arquivos do set de TREINO
# (Arquivos que já possuem todas as colunas nomeadas corretamente)
train_column_mapping = {
    'flow': 'streamflow_m3s',
    'precip_obs': 'precipitation_chirps',
    'et_obs': 'potential_evapotransp_gleam',
    'precip_forecast': 'precipitation_forecast',
    'et_forecast': 'et_forecast'
}

# Processamento de Features para Treino
combined_df = process_features(
    input_dir=root_folder / "data" / "complete_series",
    forecast_dir=root_folder / "data" / "forecast",
    output_dir=root_folder / "data" / "processed",
    station_ids=stations,
    use_config_file=True,
    output_filename="features_combined_train.csv"
)

NameError: name 'stations' is not defined

In [9]:
# Carregar configurações do arquivo YAML
config_path = CONFIG_DIR / "data_config.yaml"

try:
    # Usar o config_loader para carregar e validar
    sys.path.insert(0, str(root_folder / "src/utils"))
    from config_loader import load_feature_config, load_split_config, ConfigLoader

    sys.path.insert(0, str(root_folder / "src/data_processing"))
    from features_processing import process_features
    
    # Carregar configurações de features
    feature_config = load_feature_config(validate=True)
    
    print("✅ Configurações de features carregadas do arquivo YAML:")
    print(f"Arquivo: {config_path}")
    print()
    
    for key, value in feature_config.items():
        print(f"  - {key}: {value}")
    
    # Extrair configurações de features
    api_k_list = feature_config['api_k_list']
    precipitation_ma_windows = feature_config['precipitation_ma']
    precipitation_cumulative_windows = feature_config['precipitation_cum']
    forecast_ma_windows = feature_config.get('forecast_ma', precipitation_ma_windows)
    forecast_cumulative_windows = feature_config.get('forecast_cum', precipitation_cumulative_windows)
    evapotranspiration_ma_windows = feature_config['evapotranspiration_ma']
    anomaly_ma_windows = feature_config['anomaly_ma']
    
    # Carregar configurações de split
    split_config = load_split_config(validate=True)
    
    print("\n✅ Configurações de split carregadas:")
    for key, value in split_config.items():
        print(f"  - {key}: {value}")
    
except FileNotFoundError as e:
    print(f"❌ ERRO CRÍTICO: {e}")
    print("\nSolução:")
    print("1. Crie o arquivo data_config.yaml na pasta config/")
    print("2. Adicione as seções 'feature_windows' e 'split_config'")
    print("3. Execute novamente o notebook")
    # Parar a execução
    raise
    
except ValueError as e:
    print(f"❌ ERRO DE CONFIGURAÇÃO: {e}")
    print("\nSolução:")
    print(f"1. Edite o arquivo: {config_path}")
    print("2. Corrija as configurações conforme indicado acima")
    print("3. Execute novamente o notebook")
    raise
    
except ImportError as e:
    print(f"❌ ERRO DE IMPORT: {e}")
    print("\nSolução:")
    print("1. Verifique se o módulo config_loader existe em src/utils/")
    print("2. Verifique se o arquivo data_config.yaml existe em config/")
    raise

✅ Configurações de features carregadas do arquivo YAML:
Arquivo: c:\Users\emily\Documents\GitHub\Mini-curso-GitHub-Leo\config\data_config.yaml

  - api_k_list: [0.7, 0.8, 0.85, 0.9, 0.92, 0.95]
  - precipitation_ma: [3, 7, 15]
  - precipitation_cum: [3, 5, 7, 10]
  - evapotranspiration_ma: [7, 14, 30]
  - anomaly_ma: [3, 7]
  - forecast_ma: [3, 7, 15]
  - forecast_cum: [3, 5, 7, 10]

✅ Configurações de split carregadas:
  - train_ratio: 0.95
  - val_ratio: 0.025
  - test_ratio: 0.025
  - gap: 128
  - window_stride: 1


In [10]:
# Configuração das Estações
station_ids = [10100000, 13150000, 14100000]

# Determinar o período dos dados automaticamente
print("\n🔍 Determinando período dos dados...")
try:
    # Carregar um arquivo de exemplo para determinar datas
    sample_file = COMPLETE_SERIES_DIR / f"{station_ids[0]}_complete_date.csv"
    sample_df = pd.read_csv(sample_file)
    sample_df['date'] = pd.to_datetime(sample_df['date'])
    
    start_date = sample_df['date'].min().strftime("%Y-%m-%d")
    end_date = sample_df['date'].max().strftime("%Y-%m-%d")
    
    print(f"  Período encontrado: {start_date} a {end_date}")
    print(f"  Total de dias: {(pd.to_datetime(end_date) - pd.to_datetime(start_date)).days}")
    
except Exception as e:
    print(f"⚠️  Não foi possível determinar o período automaticamente: {e}")
    # Valores padrão como fallback
    start_date = "1990-01-01"
    end_date = "2024-12-31"
    print(f"  Usando período padrão: {start_date} a {end_date}")

# Calcular datas de split automaticamente
loader = ConfigLoader()
split_dates = loader.calculate_split_dates(start_date, end_date)

print("\n📅 Datas de split calculadas automaticamente:")
print(f"  Train: {split_dates['train_start']} → {split_dates['train_end']} ({split_dates['train_days']} dias, {split_dates['train_days']/split_dates['total_days']*100:.1f}%)")
print(f"  Val:   {split_dates['train_end']} → {split_dates['val_end']} ({split_dates['val_days']} dias, {split_dates['val_days']/split_dates['total_days']*100:.1f}%)")
print(f"  Test:  {split_dates['val_end']} → {split_dates['test_end']} ({split_dates['test_days']} dias, {split_dates['test_days']/split_dates['total_days']*100:.1f}%)")

# Definir train_date_cutoff como o final do conjunto de treino
train_date_cutoff = split_dates['train_end']

print("\n📊 Configurações de janelas por tipo de feature:")
print(f"Precipitação (observada):")
print(f"  - Médias móveis: {precipitation_ma_windows}")
print(f"  - Acumulados: {precipitation_cumulative_windows}")
print(f"\nForecast de precipitação:")
print(f"  - Médias móveis: {forecast_ma_windows}")
print(f"  - Acumulados: {forecast_cumulative_windows}")
print(f"\nEvapotranspiração:")
print(f"  - Médias móveis: {evapotranspiration_ma_windows}")
print(f"\nAnomalias:")
print(f"  - Médias móveis: {anomaly_ma_windows}")
print(f"\nAPI:")
print(f"  - Valores k: {api_k_list}")


# Chamar função de processamento
print("\n🚀 Iniciando processamento de features...")
combined_df = process_features(
    input_dir=COMPLETE_SERIES_DIR,
    forecast_dir=FORECAST_DIR,
    output_dir=PROCESSED_DIR,
    station_ids=station_ids,
    # Não precisamos passar parâmetros individuais, serão carregados do config
    api_k_list=None,
    precipitation_ma_windows=None,
    precipitation_cumulative_windows=None,
    forecast_ma_windows=None,
    forecast_cumulative_windows=None,
    evapotranspiration_ma_windows=None,
    anomaly_ma_windows=None,
    train_date_cutoff=train_date_cutoff,
    output_filename="features_combined.csv",
    use_config_file=True  # Habilitar carregamento do arquivo de configuração
)

print("\n✅ Processamento de features concluído!")
print(f"  Arquivo salvo: {PROCESSED_DIR / 'features_combined.csv'}")
print(f"  Train date cutoff usado: {train_date_cutoff}")


🔍 Determinando período dos dados...
  Período encontrado: 1995-01-01 a 2012-04-30
  Total de dias: 6329

📅 Datas de split calculadas automaticamente:
  Train: 1995-01-01 → 2011-06-18 (6012 dias, 95.0%)
  Val:   2011-06-18 → 2011-11-23 (158 dias, 2.5%)
  Test:  2011-11-23 → 2012-04-30 (159 dias, 2.5%)

📊 Configurações de janelas por tipo de feature:
Precipitação (observada):
  - Médias móveis: [3, 7, 15]
  - Acumulados: [3, 5, 7, 10]

Forecast de precipitação:
  - Médias móveis: [3, 7, 15]
  - Acumulados: [3, 5, 7, 10]

Evapotranspiração:
  - Médias móveis: [7, 14, 30]

Anomalias:
  - Médias móveis: [3, 7]

API:
  - Valores k: [0.7, 0.8, 0.85, 0.9, 0.92, 0.95]

🚀 Iniciando processamento de features...
PROCESSAMENTO DE FEATURES

📊 Carregando dados observados de 3 estações...
✓ 3 estações carregadas

🔮 Carregando dados de forecast...
✓ 3 estações com forecast carregadas

🔗 Fazendo merge observado + forecast...
✓ 3 estações combinadas

⚙️  Criando features...
✓ Estação 10100000 processada